In [1]:
# 허깅페이스와 위스퍼 모델 연결하기 (로컬 환경에서 언어 모델을 내려받아 사용하기)
# 회의 내용이 회사 밖으로 유출되면 안되거나 음성 파일의 길이가 길어 처리 비용이 걱정된다.면...

# 허깅 페이스 (인공지능 모델을 개발하는 회사 : https://huggingface.co/) -> 왼쪽 검색창에 whisper 검색
# openai/whisper-large-v3-turbo -> 페이지 아래쪽에 보면 위스퍼 모델 사용법이 있다.
# https://huggingface.co/openai/whisper-large-v3-turbo

# 허깅페이스 설치 해보기
# pip install --upgrade pip 
# pip install --upgrade transformers datasets[audio] accelerate
# 버전 충돌 해결
# pip install torch==2.3.1 torchaudio==2.3.1 torchvision==0.18.1 transformers==4.41.2 datasets==2.19.0 accelerate==0.30.1

In [2]:
# 허깅페이스의 모델을 사용하려면 코덱이 있어야 함 (FFMPEG) 설치
# https://www.ffmpeg.org/
# https://www.gyan.dev/ffmpeg/builds/ 윈도우용 다운로드

import os
os.environ["PATH"] += os.pathsep + r"C:\Aiprojects\ffmpeg\bin" # 압축푼 파일 위치를 기록한다.


In [3]:
# cpu용
# pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu
# pip uninstall torchcodec -y 코덱 충돌
# pip uninstall transformers -y 2번 실행하여 의존성까지 삭제
# pip install transformers==4.41.2 버전 낮춰 실행
# pip install librosa soundfile 코덱 연동용
# 상단에 커널 Restart

# gpu용 https://pytorch.org/ 하단에 install 부분에서 하드웨어 스팩에 맞게 선택
# pip3 install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cpu

# https://huggingface.co/openai/whisper-large-v3-turbo 하단 코드를 붙이고 수정한다.

import torch
from transformers import AutoModelForSpeechSeq2Seq, AutoProcessor, pipeline
# from datasets import load_dataset # 여기에 샘플 데이터들이 들어 있지만 현재 로컬에 있는 mp3를 사용함

# device = "cuda:0" if torch.cuda.is_available() else "cpu"
# torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

device = "cpu"
torch_dtype = torch.float16 if torch.cuda.is_available() else torch.float32

model_id = "openai/whisper-large-v3-turbo"
# model_id = "openai/whisper-small"

model = AutoModelForSpeechSeq2Seq.from_pretrained(
    model_id, torch_dtype=torch_dtype, low_cpu_mem_usage=True, use_safetensors=True
)
model.to(device)

processor = AutoProcessor.from_pretrained(model_id)

# pipeline()오디오를 처리하는데 필요한 클래스
pipe = pipeline(
    "automatic-speech-recognition",
    model=model,
    tokenizer=processor.tokenizer,
    feature_extractor=processor.feature_extractor,
    torch_dtype=torch_dtype,
    device=device,
    return_timestamps=True,   # 청크별로 타임스탬프 반환(아래 10초씩 분할)
    chunk_length_s=10,  # 입력 오디오 10초씩 나누기
    stride_length_s=2,  # 2초씩 겹치도록 청크 나누기(오디오 분석의 정확도를 높임)
) 

# dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
# sample = dataset[0]["audio"]
sample = "../audio/lsy_audio_2023_58s.mp3"

# result = pipe(sample)
# print(result["text"]) 

# print(result)
result = pipe(
    sample,
    generate_kwargs={"language": "ko"}
)

print(result["text"])

# 아래 오류 발생시 파이토치 버전 문제임 맨위에 pip 설치 진행
# 
# ---------------------------------------------------------------------------
# RuntimeError                              Traceback (most recent call last)
# Cell In[5], line 36
#      32 # dataset = load_dataset("distil-whisper/librispeech_long", "clean", split="validation")
#      33 # sample = dataset[0]["audio"]
#      34 sample = "../audio/lsy_audio_2023_58s.mp3"
#      35 
# ---> 36 result = pipe(sample)
#      37 # print(result["text"])
#      38 
#      39 print(result)

# File c:\Aiprojects\venv\Lib\site-packages\transformers\pipelines\automatic_speech_recognition.py:244, in AutomaticSpeechRecognitionPipeline.__call__(self, inputs, **kwargs)
#     187 def __call__(self, inputs: np.ndarray | bytes | str | dict, **kwargs: Any) -> list[dict[str, Any]]:
#     188     """
#     189     Transcribe the audio sequence(s) given as inputs to text. See the [`AutomaticSpeechRecognitionPipeline`]
#     190     documentation for more information.
#    (...)    242                 `"".join(chunk["text"] for chunk in output["chunks"])`.
#     243     """
# --> 244     return super().__call__(inputs, **kwargs)

# File c:\Aiprojects\venv\Lib\site-packages\transformers\pipelines\base.py:1256, in Pipeline.__call__(self, inputs, num_workers, batch_size, *args, **kwargs)
#    1254     return self.iterate(inputs, preprocess_params, forward_params, postprocess_params)
#    1255 elif isinstance(self, ChunkPipeline):
# -> 1256     return next(
# ...
#     torch.ops.load_library(core_library_path)
#   File "c:\Aiprojects\venv\Lib\site-packages\torch\_ops.py", line 1505, in load_library
#     raise OSError(f"Could not load this library: {path}") from e
# OSError: Could not load this library: C:\Aiprojects\venv\Lib\site-packages\torchcodec\libtorchcodec_core4.dll
# [end of libtorchcodec loading traceback].
# Output is truncated. View as a scrollable element or open in a text editor. Adjust cell output settings...

c:\Aiprojects\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


 안녕하세요. 이 강의는 GPT API로 챗봇 만들기 라는 내용을 다루는 강의입니다. GPT API에 대해서 생소하신 분들도 있을텐데 우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서 우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요. 그래서 이런 강의들이 사실 많이 있습니다. 그래서 여러 가지들이 있는데 이 강의 특징이라고 한다면 GPT로 명확한 미션을 달성하는 챕터 프로그램을 만드는게 사실 쉽지는 않은데 이걸 어떻게 해서 구현을 하는지 그리고 그게 왜 필요한지에 대해서 좀 이야기를 할 거고요. 그 예제로 예제는 여러가지가 될 수 있는데 여기서 예제로 하는 것은 음악 플레이리스트 동영상을 자동으로 대화를 통해서 생성하는 프로그램을 만드는 것을 다루려고 합니다. 그래서 프로그램이 실행되는 모습을 한번 보여드릴게요. 우리가 만들 프로그램은 이런 식으로 이제 나타나게 되고


In [4]:
# chunks를 CSV 파일로 저장
start_end_text = []

for chunk in result["chunks"]:
    start = chunk["timestamp"][0]
    end = chunk["timestamp"][1]
    text = chunk["text"]
    start_end_text.append([start, end, text])

import pandas as pd
df = pd.DataFrame(start_end_text, columns=["start", "end", "text"])
df.to_csv("lsy_audio_2023_58.csv", index=False, sep="|")
display(df)

,start,end,text
0,0.00,6.30,안녕하세요. 이 강의는 GPT API로 챗봇 만들기 라는 내용을 다루는 강의입니다.
1,7.18,10.00,GPT API에 대해서 생소하신 분들도 있을텐데
2,11.00,17.00,"우리가 잘 알고 있는 ChatGPT, ChatGPT 기능을 이용해서"
3,17.00,21.00,우리가 원하는 프로그램을 어떻게 만드는지에 대해서 이야기할 거예요.
4,21.00,24.00,그래서 이런 강의들이 사실 많이 있습니다.
5,24.00,27.48,그래서 여러 가지들이 있는데 이 강의 특징이라고 한다면
6,27.48,29.58,GPT로 명확한 미션을 달성하는
7,29.58,31.66,챕터 프로그램을 만드는게 사실
8,31.66,34.32,쉽지는 않은데 이걸 어떻게 해서
9,34.32,36.40,구현을 하는지 그리고 그게 왜 필요한지에 대해서
